# SWE-bench Live — Dataset Exploration

MSc research project: investigating whether codebase maps improve AI coding agent performance.

In [1]:
from datasets import load_dataset
import pandas as pd
import json

/home/afb225/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the dataset

In [15]:
ds = load_dataset('SWE-bench-Live/SWE-bench-Live', split='full')
df = ds.to_pandas()
print('Shape:', df.shape)
print('\nColumns:')
for col in df.columns:
    print(f'  {col}')

Shape: (1888, 18)

Columns:
  repo
  pull_number
  instance_id
  issue_numbers
  base_commit
  patch
  test_patch
  problem_statement
  hints_text
  all_hints_text
  commit_urls
  created_at
  commit_url
  test_cmds
  log_parser
  difficulty
  FAIL_TO_PASS
  PASS_TO_PASS


## 2. Sample row — full view of all fields

In [16]:
pd.set_option('display.max_colwidth', None)
sample = df.iloc[0]
for field, value in sample.items():
    print(f'=== {field} ===')
    print(value)
    print()

=== repo ===
python-babel/babel

=== pull_number ===
1120

=== instance_id ===
python-babel__babel-1120

=== issue_numbers ===
['1078']

=== base_commit ===
0005c85fccccb491930f064ea8d064e87ce7ee79

=== patch ===
diff --git a/babel/messages/pofile.py b/babel/messages/pofile.py
index ec2d57f7d..721707657 100644
--- a/babel/messages/pofile.py
+++ b/babel/messages/pofile.py
@@ -637,8 +637,8 @@ def generate_po(
     # provide the same behaviour
     comment_width = width if width and width > 0 else 76
 
-    comment_wrapper = TextWrapper(width=comment_width)
-    header_wrapper = TextWrapper(width=width, subsequent_indent="# ")
+    comment_wrapper = TextWrapper(width=comment_width, break_long_words=False)
+    header_wrapper = TextWrapper(width=width, subsequent_indent="# ", break_long_words=False)
 
     def _format_comment(comment, prefix=''):
         for line in comment_wrapper.wrap(comment):
diff --git a/babel/util.py b/babel/util.py
index 180afa093..e0b4e2a61 100644
--- a/babel/util

## 3. Unique repos and issue counts

In [17]:
# Try common column names for repo
repo_col = next((c for c in df.columns if 'repo' in c.lower()), None)
print(f'Using column: {repo_col}')

repo_counts = df[repo_col].value_counts().reset_index()
repo_counts.columns = ['repo', 'count']
print(f'\nUnique repos: {len(repo_counts)}')
print()
print(repo_counts.to_string(index=False))

Using column: repo

Unique repos: 223

                                             repo  count
                                   conan-io/conan    165
                      aws-cloudformation/cfn-lint    109
                            matplotlib/matplotlib    102
                              deepset-ai/haystack     88
                                pylint-dev/pylint     62
                          instructlab/instructlab     52
                                 keras-team/keras     48
                                reflex-dev/reflex     44
                            streamlink/streamlink     41
                                sphinx-doc/sphinx     39
                                  pdm-project/pdm     35
                            sissbruecker/linkding     34
                               pvlib/pvlib-python     30
                                    pydata/xarray     29
                                beeware/briefcase     28
                                  kedro-org/kedro

## 4. Date range of issues

In [18]:
# Find date column(s)
date_cols = [c for c in df.columns if any(kw in c.lower() for kw in ['date', 'created', 'time', 'timestamp'])]
print('Date-like columns:', date_cols)

for col in date_cols:
    parsed = pd.to_datetime(df[col], errors='coerce', utc=True)
    if parsed.notna().any():
        print(f'\n[{col}]')
        print(f'  Earliest: {parsed.min()}')
        print(f'  Latest:   {parsed.max()}')
        print(f'  Span:     {parsed.max() - parsed.min()}')

Date-like columns: ['created_at']

[created_at]
  Earliest: 2021-07-22 13:35:28+00:00
  Latest:   2025-09-02 09:55:04+00:00
  Span:     1502 days 20:19:36


In [19]:
for split in ['test', 'train', 'full']:
    try:
        ds = load_dataset('SWE-bench-Live/SWE-bench-Live', split=split)
        df = ds.to_pandas()
        print(f'{split}: {len(df)} issues, latest: {df["created_at"].max()}')
    except:
        print(f'{split}: not available')

test: 1000 issues, latest: 2025-08-30 07:39:31
train: not available
full: 1888 issues, latest: 2025-09-02 09:55:04


## 5. Save repo issue counts to JSON

In [ ]:
counts_dict = repo_counts.set_index('repo')['count'].to_dict()

with open('swe_live_repo_counts.json', 'w') as f:
    json.dump(counts_dict, f, indent=2)

print('Saved swe_live_repo_counts.json')
print(json.dumps(counts_dict, indent=2))